In [2]:
from urllib.request import urlopen

from IPython.core.display import Markdown
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

from utils.std_model import base_model

checkpointer = MemorySaver()
backend = StateBackend()
llm = base_model()

skill_url = "https://raw.githubusercontent.com/langchain-ai/deepagents/refs/heads/main/libs/cli/examples/skills/langgraph-docs/SKILL.md"
with urlopen(skill_url) as response:
    skill_content = response.read().decode('utf-8')

skills_files = {
    "/skills/langgraph-docs/SKILL.md": create_file_data(skill_content),
}

agent = create_deep_agent(
    model=llm,
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer,
    middleware=[CodeInterpreterMiddleware(skills_backend=backend)],  # for interpreter skills
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What is langgraph?"}],
        # Seed the default StateBackend's in-state filesystem (virtual paths must start with "/").
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "12345"}},
)

Markdown(result['messages'][-1].content)

## What is LangGraph?

**LangGraph** is a framework within the LangChain ecosystem for building **stateful, multi-actor applications** with LLMs. Instead of simple linear chains, it represents agent workflows as **directed graphs** — giving you fine-grained control over state, branching, loops, and multi-agent coordination.

### Key Concepts

- **StateGraph** — The core abstraction. You define nodes (computational steps) and edges (control flow), with a shared **state** object flowing through the graph.
- **State** — A structured schema (TypedDict or Pydantic model) that every node reads from and writes to. This makes the agent's internal state explicit and inspectable.
- **Nodes** — Any Python function or Runnable. Takes state in, returns state updates.
- **Edges** — Normal (always transition), conditional (route based on state logic), and entry/exit points.
- **Human-in-the-Loop** — Pause execution at any node for human review, approval, or edits.
- **Persistence & Checkpointing** — Save execution state at each step for fault tolerance, interruptions, and replay.
- **Multi-Agent Systems** — Coordinate multiple agents in a single graph (e.g., supervisor delegates to workers, agents debate/negotiate).

### When to Use It

| Use Case | Why LangGraph |
|---|---|
| Complex agent loops with branching/retry logic | Graph structure handles non-linear flows naturally |
| Multi-agent orchestration | Supervisor/worker patterns built-in |
| Workflows needing human approval | Pause/resume at any node |
| Long-running agents | Persistence and checkpointing |
| ReAct agents | The classic agent loop is a trivial graph |

### Minimal Example

```python
from langgraph.graph import StateGraph
from typing import TypedDict, list

class AgentState(TypedDict):
    messages: list

def call_model(state):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.set_entry_point("agent")
graph.add_edge("agent", "__end__")
app = graph.compile()
```

For the full documentation, check out [langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/).

In [3]:

from pathlib import Path
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver
from utils.std_model import base_model

llm = base_model()

# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()
backend = FilesystemBackend(root_dir=Path(".").resolve(), virtual_mode=True)

agent = create_deep_agent(
    model=llm,
    backend=backend,
    skills=["skills/"],
    checkpointer=checkpointer,  # Required!
    middleware=[CodeInterpreterMiddleware(skills_backend=backend)],  # for interpreter skills
)

content = "我想看文章，请问有哪些文章并指出路径。然后在文章文件的最下面使用写入工具写入一句话 “你好呀，柯梦剑” "
# content = "我想看文章，请问有哪些文章并指出路径。"
# content = "我想看《勇敢的小蚂蚁》"

result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "12345"}},
)

result

{'messages': [HumanMessage(content='我想看文章，请问有哪些文章并指出路径。然后在文章文件的最下面使用写入工具写入一句话 “你好呀，柯梦剑” ', additional_kwargs={}, response_metadata={}, id='7437998c-50a8-4afb-8888-1a791530c5bd'),
  AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants me to:\n1. Show what articles are available and their paths\n2. Write "你好呀，柯梦剑" at the bottom of the article files\n\nLet me first look at the `read-doc` skill to understand how to handle this, and also look around the filesystem for articles.\n\nLet me start by reading the skill file and exploring the filesystem for articles.'}, response_metadata={'token_usage': {'completion_tokens': 200, 'prompt_tokens': 6759, 'total_tokens': 6959, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 81, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 6656}, 'prompt_cache_hit_tokens': 6656, 'prompt_cache_miss_tokens': 103}